# Data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from datetime import datetime, timedelta

import requests
import time
import glob
import os
import re

from IPython.display import display

In [2]:
import plotly.io as pio
pio.renderers.default = "browser" # notebook

In [3]:
offset = 0
today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

dtype_map = {
    "agg_pjm_ExternalNodeID": "int64",
    "pjm_ExternalNodeID": "int64",
    "agg_NodeName": "string",
    "NodeName": "string",
    "forecast_area": "string",
    "is_valid": "int8"
}

hourly_load = pd.read_csv(
    f"data/hourly_load_MW_by_distribution/hourly_load_MW_by_distribution_{today_str}.csv",
    skiprows=[1],
    dtype=dtype_map,
    parse_dates=["datetime_ending_ept", "insert_datetime"]
)

hourly_load = hourly_load.sort_values("datetime_ending_ept").reset_index(drop=True)

# PJM update nodename
hourly_load = hourly_load[hourly_load["datetime_ending_ept"] >= "2026-03-12"].copy()

In [4]:
node_cty_map = pd.read_csv('data/pnode_county_mapping.csv')
node_cty_map.columns = (
    node_cty_map.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [5]:
hourly_load.columns = (
    hourly_load.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

hourly_load = hourly_load.drop(columns=[
    "autokey",
    "insert_datetime"
], errors="ignore")

hourly_load["distribution_factor"] = (
    hourly_load["distribution_factor"]
    .astype(float)
    .round(8)
)

hourly_load = hourly_load.dropna(subset=[
    "datetime_ending_ept",
    "nodename",
    "distributed_load_mw"
])

hourly_load = hourly_load.drop_duplicates(
    subset=["datetime_ending_ept", "nodename"]
)


In [6]:
hourly_load.head()

,datetime_ending_ept,agg_pjm_externalnodeid,agg_nodename,pjm_externalnodeid,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
0,2026-04-01,8445784,AEP,32411713,JOHNSONM138 KV T1,-0.00004,AEP,15108.0,-0.58921,1
1,2026-04-01,34964545,DOM,34886335,OAKGROV435 KV TX1,0.00006,DOMINION,14019.0,0.85516,1
2,2026-04-01,34964545,DOM,1183224859,OAKGROV435 KV TX2,0.00052,DOMINION,14019.0,7.23380,1
3,2026-04-01,34964545,DOM,34885947,OAKRIDGE13 KV TX1,0.00037,DOMINION,14019.0,5.24311,1
4,2026-04-01,34964545,DOM,1305122802,OAKRIDGE13 KV TX2,0.00023,DOMINION,14019.0,3.23839,1


In [7]:
folder = "//GC-NAS/QuantTeam/cshi/XDAI_daily_constr"

files = sorted(
    glob.glob(os.path.join(folder, "DART_constr_*_morning.csv")) +
    glob.glob(os.path.join(folder, "DART_constr_*_afternoon.csv"))
)

# make sure same-date morning comes before afternoon
def file_sort_key(f):
    base = os.path.basename(f)
    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")
    run_type = m.group(2)
    run_order = {"morning": 0, "afternoon": 1}[run_type]
    return file_dt, run_order

files = sorted(files, key=file_sort_key)

dfs = []
last_file = files[-1]

for f in files:
    base = os.path.basename(f)

    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")

    df = pd.read_csv(f)

    # identify contiguous RT chunks
    is_rt = df["MARKET_TYPE_CODE"].eq("RT")
    chunk_id = (is_rt & ~is_rt.shift(fill_value=False)).cumsum()

    df_rt = df[is_rt].copy()
    df_rt["rt_chunk"] = chunk_id[is_rt].values

    # RT chunks are numbered only among RT chunks
    rt_chunk_map = {
        old: new
        for new, old in enumerate(sorted(df_rt["rt_chunk"].unique()), start=1)
    }
    df_rt["rt_chunk"] = df_rt["rt_chunk"].map(rt_chunk_map)

    # keep first RT chunk from every file
    df_keep = df_rt[df_rt["rt_chunk"].eq(1)].copy()
    df_keep.insert(0, "file_date", (file_dt - pd.Timedelta(days=1)).date())

    dfs.append(df_keep)

    # also keep second RT chunk from the latest file as today's data
    if f == last_file:
        df_today = df_rt[df_rt["rt_chunk"].eq(2)].copy()
        df_today.insert(0, "file_date", file_dt.date())
        dfs.append(df_today)

RT_constr = pd.concat(dfs, ignore_index=True)

RT_constr = RT_constr.drop(columns="rt_chunk")

# remove duplicates, keeping the later-read file
# afternoon overwrites morning when same records exist
dedup_cols = [c for c in RT_constr.columns]

RT_constr = (
    RT_constr
    .drop_duplicates(subset=dedup_cols, keep="last")
    .reset_index(drop=True)
)

RT_constr.head()

,file_date,MARKET_TYPE_CODE,MONITORED_FACILITY_DESC,CONTINGENCY_FACILITY_DESC,SHADOW_PRICE,INTF_Name,IsNew,1,2,3,...,16,17,18,19,20,21,22,23,24,VALUE_DATETIME
0,2026-03-31,RT,DUMONT2-STILLWEL TIE A 345 KV,112 WILT 765 KV 112 WILTO 65BT3-4 CB;112 WILT ...,-970.359985,DUMONT2-STILLWEL TIE A 345 KV L/O 112 ...,New,NaN,NaN,NaN,...,88.12,50.21,169.32,93.85,4.71,NaN,NaN,NaN,NaN,2026-03-31 14:15:00
1,2026-03-31,RT,MARDELA-VIENNA 6708-1 B 69 KV,LINE 138 KV LORETTO-VIENNA 13780;LORETTO 138 K...,-952.409973,MARDELA-VIENNA 6708-1 B 69 KV L/O LINE ...,New,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,79.37,NaN,NaN,NaN,NaN,2026-03-31 19:30:00
2,2026-03-31,RT,OLIVE 2-P XFORMER H 345 KV,COOK 345 KV COOK M CB;COOK 345 ...,-682.669983,OLIVE 2-P XFORMER H 345 KV L/O COOK...,New,NaN,NaN,NaN,...,395.95,43.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-31 15:25:00
3,2026-03-31,RT,155 NELS-15508 15508 1 A 138 KV,155 NELS 345 KV 155 NELSO 45BT1-10 CB;155 NELS...,-568.640015,155 NELS-15508 15508 1 A 138 KV L/O 155 ...,New,NaN,NaN,NaN,...,133.49,168.02,35.02,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-31 14:15:00
4,2026-03-31,RT,94 HAURD-11323 11323 A 138 KV,107 DIXO 138 KV 107 DIXON 38L10714 CB;169 MCGI...,-44.290001,94 HAURD-11323 11323 A 138 KV L/O 107 ...,New,NaN,17.34,29.96,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-31 01:30:00


In [8]:
folder = "//GC-NAS/QuantTeam/cshi/XDAI_daily_constr"

files = sorted(
    glob.glob(os.path.join(folder, "DART_constr_*_morning.csv")) +
    glob.glob(os.path.join(folder, "DART_constr_*_afternoon.csv"))
)

# make sure same-date morning comes before afternoon
def file_sort_key(f):
    base = os.path.basename(f)
    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")
    run_type = m.group(2)
    run_order = {"morning": 0, "afternoon": 1}[run_type]
    return file_dt, run_order

files = sorted(files, key=file_sort_key)

dfs = []

for f in files:
    base = os.path.basename(f)

    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")

    df = pd.read_csv(f)

    # identify contiguous DA chunks
    is_da = df["MARKET_TYPE_CODE"].eq("DA")
    chunk_id = (is_da & ~is_da.shift(fill_value=False)).cumsum()

    df_da = df[is_da].copy()
    df_da["da_chunk"] = chunk_id[is_da].values

    # DA chunks are numbered only among DA chunks
    da_chunk_map = {
        old: new
        for new, old in enumerate(sorted(df_da["da_chunk"].unique()), start=1)
    }
    df_da["da_chunk"] = df_da["da_chunk"].map(da_chunk_map)

    # keep first DA chunk from every file
    df_keep = df_da[df_da["da_chunk"].eq(1)].copy()
    
    run_type = m.group(2)

    if run_type == "morning":
        file_date_val = file_dt
    else:  # afternoon
        file_date_val = file_dt + pd.Timedelta(days=1)

    df_keep.insert(0, "file_date", file_date_val.date())

    dfs.append(df_keep)

DA_constr = pd.concat(dfs, ignore_index=True)

DA_constr = DA_constr.drop(columns="da_chunk")

# remove duplicates (afternoon overwrites morning)
# dedup_cols = [c for c in DA_constr.columns]

dedup_cols = [
    "file_date",
    "MARKET_TYPE_CODE",
    "MONITORED_FACILITY_DESC",
    "CONTINGENCY_FACILITY_DESC",
    "INTF_Name"
]

DA_constr = (
    DA_constr
    .drop_duplicates(subset=dedup_cols, keep="last")
    .reset_index(drop=True)
)

DA_constr.head()

,file_date,MARKET_TYPE_CODE,MONITORED_FACILITY_DESC,CONTINGENCY_FACILITY_DESC,SHADOW_PRICE,INTF_Name,IsNew,1,2,3,...,16,17,18,19,20,21,22,23,24,VALUE_DATETIME
0,2026-04-01,DA,BED-BLA contingency 86,TEMP:L500.Bismark-Doubs & Doubs.500.Cap4,-102.919998,BED-BLA contingency 86 L/O TEMP:L500.Bismark-D...,New,NaN,NaN,NaN,...,36.85,61.14,102.92,20.37,63.73,10.79,NaN,NaN,NaN,NaN
1,2026-04-01,DA,METUCHEN_230KVMET-PIEH_1_LN,ACTUAL,-88.050003,METUCHEN_230KVMET-PIEH_1_LN L/O ACTUAL,New,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-01,DA,HAVILAND_69KVHAV-PAU_1_LN,138/69.EastLima.T3,-76.940002,HAVILAND_69KVHAV-PAU_1_LN L/O 138/69.EastLima.T3,New,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-01,DA,HOOVERSV_115KVHOO-TOW_1_LN,ACTUAL,-41.130001,HOOVERSV_115KVHOO-TOW_1_LN L/O ACTUAL,New,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,12.70,41.13,0.21,NaN,NaN
4,2026-04-01,DA,MTSTORM4_500KVMTS-PRU_1_LN,L500.Bedington-BlackOak,-36.810001,MTSTORM4_500KVMTS-PRU_1_LN L/O L500.Bedington-...,New,NaN,NaN,NaN,...,NaN,NaN,NaN,36.81,15.10,NaN,NaN,NaN,NaN,NaN


# Cloud

## shadow

In [ ]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("CLOUD")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cloud_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cloud_shadow_price = cloud_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
cloud_shadow_price["hour_ending"] = cloud_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
cloud_shadow_price["datetime_ending_ept"] = (
    cloud_shadow_price["file_date"] + pd.to_timedelta(cloud_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
cloud_shadow_price = cloud_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
cloud_shadow_price = cloud_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cloud_shadow_price.head()

In [ ]:
cloud_shadow_price['nodename'].unique()

In [ ]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("CLOUD")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cloud_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cloud_shadow_price_DA = cloud_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
cloud_shadow_price_DA["hour_ending"] = cloud_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
cloud_shadow_price_DA["datetime_ending_ept"] = (
    cloud_shadow_price_DA["file_date"] + 
    pd.to_timedelta(cloud_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
cloud_shadow_price_DA = cloud_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
cloud_shadow_price_DA = cloud_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cloud_shadow_price_DA.head()

In [ ]:
cloud_shadow_price_DA['nodename'].unique()

In [ ]:
cloud_shadow_price["shadow_price"].max()

## county

In [ ]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [ ]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [ ]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [ ]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [ ]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    cloud_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [ ]:
county_load_shadow_price.head()

In [ ]:
# county_load_shadow = county_load_shadow_price[ (county_load_shadow_price["shadow_price"].notna()) & (county_load_shadow_price["shadow_price"] != 0) ].copy()

# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [ ]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [ ]:
county_load_shadow_price = county_load_shadow_price.merge(
    cloud_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [ ]:
df = county_load_shadow_price.copy()

node = "VA_Mecklenburg"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     n1 = "VA_Mecklenburg"
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [ ]:
# node1 = "VA_Mecklenburg" 
# node2 = "NC_Gates" 

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node

In [ ]:
# FINNEYWD, ALLENCRK, BRIERY, EASTERS, BOYDTNDP, KERRDAM
# node_cty_map["nodename"] = node_cty_map["nodename"].str.replace("_", "", regex=False)
# node_cty_map[node_cty_map["nodename"].str.contains("CLOUD|FINNEYWD|ALLENCRK|BRIERY|EASTERS|BOYDTNDP|KERRDAM")]

### hourly load

In [ ]:
hourly_load.head()

In [ ]:
# cloud_hourly_load =hourly_load[hourly_load["nodename"].str.contains("CLOUD|FINNEYWD|ALLENCRK|BRIERY|EASTERS|BOYDTNDP|KERRDAM")]
cloud_hourly_load = hourly_load[
    (hourly_load["agg_nodename"] == "DOM") &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 20
    )
].copy()

# cloud_hourly_load["nodename"] = cloud_hourly_load["nodename"].str.replace("_", "", regex=False)
cloud_hourly_load.head()

In [ ]:
cloud_hourly_load["nodename"].unique()

In [ ]:
cloud_hourly_load = cloud_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

In [ ]:
cloud_hourly_load.tail()

In [ ]:
# counties_to_consider = ["VA_Mecklenburg", "VA_Brunswick", "VA_Fluvanna", "VA_Halifax", "VA_Charlotte", "VA_Lunenburg", "NC_Warren", "VA_Dinwiddie", "VA_Prince Edward"]

# cloud_hourly_load = cloud_hourly_load[cloud_hourly_load["county_name"].isin(counties_to_consider)].copy()

### weather features

In [ ]:
# node_loc = (
#     cloud_hourly_load[["nodename", "latitude", "longitude"]]
#     .rename(columns={"latitude": "lat", "longitude": "lon"})
#     .dropna()
#     .drop_duplicates()
# )

In [ ]:
# HOURLY_VARS = ["temperature_2m", "apparent_temperature", "dew_point_2m", "relative_humidity_2m", "precipitation", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "global_tilted_irradiance"]
# URL = "https://archive-api.open-meteo.com/v1/archive"

# def get_weather(lat, lon, start_date, end_date):
#     params = {
#         "latitude": lat,
#         "longitude": lon,
#         "start_date": start_date,
#         "end_date": end_date,
#         "hourly": ",".join(HOURLY_VARS),
#         "timezone": "America/New_York"
#     }
    
#     r = requests.get(URL, params=params, timeout=60)
#     r.raise_for_status()
#     data = r.json().get("hourly", {})
    
#     df = pd.DataFrame({
#         "datetime_ending_ept": pd.to_datetime(data.get("time", []))
#     })
    
#     for v in HOURLY_VARS:
#         df[v] = data.get(v)
    
#     df["lat"] = lat
#     df["lon"] = lon
    
#     return df

# weather_list = []

# start_time = cloud_hourly_load["datetime_ending_ept"].min().date()
# end_time = cloud_hourly_load["datetime_ending_ept"].max().date() - pd.Timedelta(days=1)

# for _, row in node_loc.iterrows():
#     try:
#         df = get_weather(row["lat"], row["lon"], start_time, end_time)
#         weather_list.append(df)
#         time.sleep(0.3)  # avoid rate limit
#     except Exception as e:
#         print(f"Failed for {row['nodename']}: {e}")

# weather_df = pd.concat(weather_list, ignore_index=True)

# weather_df = weather_df.rename(columns={"time": "datetime_ending_ept", "lat": "latitude", "lon": "longitude"})


In [ ]:
# cloud_load_weather = cloud_hourly_load.merge(
#     weather_df,
#     on=["latitude", "longitude", "datetime_ending_ept"],
#     how="left"
# )

In [ ]:
cloud_load_weather = cloud_hourly_load.copy()

In [ ]:
cloud_load_weather = cloud_load_weather.sort_values("datetime_ending_ept").reset_index(drop=True)
cloud_load_weather.head()

In [ ]:
cloud_load_weather = cloud_load_weather[(cloud_load_weather["datetime_ending_ept"] >= "2026-04-01")].copy()

### shadow prices

In [ ]:
cloud_load_weather["datetime_ending_ept"] = pd.to_datetime(cloud_load_weather["datetime_ending_ept"])

cloud_merged = cloud_load_weather.merge(
    cloud_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

cloud_merged["shadow_price"] = cloud_merged["shadow_price"].fillna(0)

# row-level filter
cloud_shadow = cloud_merged[
    (cloud_merged["shadow_price"].notna()) &
    (cloud_merged["shadow_price"] != 0)
].copy()


In [ ]:
cloud_merged.tail()

### correlation

In [ ]:
cloud_merged[cloud_merged['nodename'].str.contains('CLOUD')].head()

In [ ]:
# get CLOUD node center
cloud_center = (
    cloud_merged.loc[cloud_merged["nodename"].str.contains("CLOUD", na=False), ["latitude", "longitude"]]
    .drop_duplicates()
    .iloc[0]
)

lat_center = cloud_center["latitude"]
lon_center = cloud_center["longitude"]

# 2x2 degree grid around CLOUD
cloud_shadow = cloud_shadow[
    (cloud_shadow["latitude"].between(lat_center - 1, lat_center + 1)) &
    (cloud_shadow["longitude"].between(lon_center - 1, lon_center + 1))
].copy()


In [ ]:
df = cloud_shadow.copy()

# # remove shadow_price itself from feature list
# weather_cols = ["temperature_2m", "apparent_temperature", "dew_point_2m", "relative_humidity_2m", "precipitation", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "global_tilted_irradiance"]

# feature_cols = ["distribution_factor", "forecast_load_mw", "distributed_load_mw"] + weather_cols

feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
      .rename(columns={"level_1": "feature", 0: "correlation"})
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:
df_plot = cloud_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "CLOUD   230 KV  LD1U25"

# node = "NEAST   35 KV   TX6"

df = cloud_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
# df = cloud_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr


In [ ]:
# node1 = "HERBERT213.8 KV TX1LOAD"
# node2 = "SHOCKOE 35 KV   TX4"

# sub = cloud_merged[
#     cloud_merged["nodename"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = cloud_merged[
#     cloud_merged["nodename"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

# Chicago

Chicago - Praxair3 138 kV l/o Wilton Center

## shadow

In [12]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("Chicago")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
chicago_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
chicago_shadow_price = chicago_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
chicago_shadow_price["hour_ending"] = chicago_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
chicago_shadow_price["datetime_ending_ept"] = (
    chicago_shadow_price["file_date"] + pd.to_timedelta(chicago_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
chicago_shadow_price = chicago_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
chicago_shadow_price = chicago_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

chicago_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-01 04:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,16.99
1,2026-04-01 05:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,159.08
2,2026-04-01 06:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,24.06
3,2026-04-01 18:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,281.11
4,2026-04-01 19:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,159.77


In [13]:
chicago_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [14]:
# Get all nodenames that belong to agg_nodename == "COMED" in hourly_load
ce_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "COMED", "nodename"]
    .dropna()
    .unique()
)

# Map those COMED nodenames to county_name using node_cty_map
ce_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(ce_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those COMED nodenames
ce_counties = ce_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_ce_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(ce_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_ce_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_ce_counties)
].copy()

In [15]:
hourly_load_county = hourly_load_ce_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

county_load = county_load[county_load["state_short_name"] == "IL"].copy()

In [16]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [17]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [18]:
county_load.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude
0,2026-04-01,IL_Boone,IL,110.37109,42.29,-88.89
1,2026-04-01,IL_Bureau,IL,7.32962,41.53,-89.71
2,2026-04-01,IL_Cook,IL,4873.16736,41.88,-87.80
3,2026-04-01,IL_DeKalb,IL,182.88795,41.89,-88.76
4,2026-04-01,IL_DuPage,IL,1087.61351,41.84,-88.09


In [19]:
# node_loc = (
#     county_load[["county_name", "latitude", "longitude"]]
#     .rename(columns={"latitude": "lat", "longitude": "lon"})
#     .dropna()
#     .drop_duplicates()
# )

# HOURLY_VARS = ["wind_speed_10m"]
# URL = "https://archive-api.open-meteo.com/v1/archive"

# def get_weather(lat, lon, start_date, end_date):
#     params = {
#         "latitude": lat,
#         "longitude": lon,
#         "start_date": start_date,
#         "end_date": end_date,
#         "hourly": ",".join(HOURLY_VARS),
#         "timezone": "America/New_York"
#     }
    
#     r = requests.get(URL, params=params, timeout=60)
#     r.raise_for_status()
#     data = r.json().get("hourly", {})
    
#     df = pd.DataFrame({
#         "datetime_ending_ept": pd.to_datetime(data.get("time", []))
#     })
    
#     for v in HOURLY_VARS:
#         df[v] = data.get(v)
    
#     df["lat"] = lat
#     df["lon"] = lon
    
#     return df

# weather_list = []

# start_time = county_load["datetime_ending_ept"].min().date()
# end_time = county_load["datetime_ending_ept"].max().date() - pd.Timedelta(days=1)

# for _, row in node_loc.iterrows():
#     try:
#         df = get_weather(row["lat"], row["lon"], start_time, end_time)
#         weather_list.append(df)
#         time.sleep(0.01)  # avoid rate limit
#     except Exception as e:
#         print(f"Failed for {row['nodename']}: {e}")

# weather_df = pd.concat(weather_list, ignore_index=True)

# weather_df = weather_df.rename(columns={"time": "datetime_ending_ept", "lat": "latitude", "lon": "longitude"})

# county_load_weather = county_load.merge(
#     weather_df,
#     on=["latitude", "longitude", "datetime_ending_ept"],
#     how="left"
# )


In [20]:
# county_load = county_load_weather.copy()

In [21]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    chicago_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [22]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [23]:
df = county_load_shadow.copy()

# feature_cols = ["distributed_load_mw", "wind_speed_10m"] 
feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
15,IL_Whiteside,0.215303
8,IL_LaSalle,0.017718
10,IL_Lee,0.005258
4,IL_Grundy,-0.007573
2,IL_DeKalb,-0.013859
11,IL_Livingston,-0.014406
14,IL_Stephenson,-0.014734
1,IL_Cook,-0.040426
13,IL_Ogle,-0.062235
9,IL_Lake,-0.066638


In [24]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [25]:
county_load_shadow_price.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude,shadow_price
0,2026-04-01,IL_Boone,IL,110.37109,42.29,-88.89,0.0
1,2026-04-01,IL_Bureau,IL,7.32962,41.53,-89.71,0.0
2,2026-04-01,IL_Cook,IL,4873.16736,41.88,-87.80,0.0
3,2026-04-01,IL_DeKalb,IL,182.88795,41.89,-88.76,0.0
4,2026-04-01,IL_DuPage,IL,1087.61351,41.84,-88.09,0.0


In [26]:
df = county_load_shadow_price.copy()

node = "IL_Cook"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [27]:
df = county_load_shadow_price.copy()

node = "IL_Cook"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["wind_speed_10m"],
        name="Wind Speed (m/s)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Wind Speed vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Wind Speed (m/s)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

KeyError: 'wind_speed_10m'

In [ ]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [ ]:
# node1 = "IL_Boone"
# node2 = "IL_Lee"

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node 

In [ ]:
chicago_hourly_load = hourly_load[
    (hourly_load["agg_nodename"] == "COMED") &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 20
    )
].copy()

In [ ]:
chicago_hourly_load = chicago_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

In [ ]:
chicago_hourly_load.tail()

,datetime_ending_ept,agg_nodename,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,county_name,latitude,longitude,state_short_name
239054,2026-05-09,COMED,J390 138 KV TR787934,0.00200,COMED,8975.0,17.90513,IL_Will,41.45623,-88.20747,IL
239055,2026-05-09,COMED,K319-2 138 KV TR71 12,0.00286,COMED,8975.0,25.67747,IL_Kankakee,41.23823,-87.68813,IL
239056,2026-05-09,COMED,K319-2 138 KV TR72 12,0.00291,COMED,8975.0,26.15315,IL_Kankakee,41.23823,-87.68813,IL
239057,2026-05-09,COMED,K323 138 KV TR74 12,0.00108,COMED,8975.0,9.66608,IL_Kankakee,41.13853,-87.89635,IL
239058,2026-05-09,COMED,85 SKOKI138 KV TR1 34,0.00106,COMED,8975.0,9.47760,IL_Cook,42.02415,-87.74026,IL


In [ ]:
chicago_hourly_load = chicago_hourly_load[(chicago_hourly_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [ ]:
chicago_merged = chicago_hourly_load.merge(
    chicago_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

chicago_merged["shadow_price"] = chicago_merged["shadow_price"].fillna(0)

# total unique timestamps
total_ts = chicago_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    chicago_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
chicago_merged = chicago_merged[chicago_merged['nodename'].isin(valid_nodes)]

# row-level filter
chicago_shadow = chicago_merged[
    (chicago_merged["shadow_price"].notna()) &
    (chicago_merged["shadow_price"] != 0)
].copy()

In [ ]:
df = chicago_shadow.copy()

feature_cols = ["distributed_load_mw"]

# correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,nodename,distributed_load_mw
243,H471 345 KV TR83 34,0.253695
249,J390 138 KV TR767734,0.248306
26,126 STAT138 KV TR75 12,0.239608
23,126 STAT138 KV TR72 12,0.190135
257,W541 GFX138 KV TR5,0.157649
...,...,...
137,51 MC CO138 KV TR77 34,-0.268202
130,454 PLAI138 KV TR73 12,-0.271782
201,78 FRANK34.5 KV TR78,-0.272804
197,78 FRANK138 KV TR73 12,-0.275252


In [ ]:

df_plot = chicago_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "453 WOOD138 KV  TR71 12"

df = chicago_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
# df = chicago_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] + pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr


In [ ]:
# # April 12th: both -55
# node1 = "13 CRAWF138 KV  TR72 12" 
# node2 = "13 CRAWF138 KV  TR73 12"

# sub = chicago_merged[
#     chicago_merged["nodename"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = chicago_merged[
#     chicago_merged["nodename"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

# BURNHAM


## shadow

In [28]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("BURNHAM")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
burnham_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
burnham_shadow_price = burnham_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
burnham_shadow_price["hour_ending"] = burnham_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
burnham_shadow_price["datetime_ending_ept"] = (
    burnham_shadow_price["file_date"] + pd.to_timedelta(
        burnham_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
burnham_shadow_price = burnham_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
burnham_shadow_price = burnham_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

burnham_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-03-31 02:00:00,BURNHAM-MUNSTER2 NIP TIE A 345 KV,5.48
1,2026-03-31 03:00:00,BURNHAM-MUNSTER2 NIP TIE A 345 KV,11.62
2,2026-03-31 04:00:00,BURNHAM-MUNSTER2 NIP TIE A 345 KV,45.32
3,2026-03-31 05:00:00,BURNHAM-MUNSTER2 NIP TIE A 345 KV,2.84
4,2026-03-31 06:00:00,BURNHAM-MUNSTER2 NIP TIE A 345 KV,6.58


In [29]:
burnham_shadow_price["shadow_price"].max()

np.float64(925.6)

In [30]:
full_range = pd.date_range(
    start=min(
        chicago_shadow_price["datetime_ending_ept"].min(),
        burnham_shadow_price["datetime_ending_ept"].min()
    ),
    end=max(
        chicago_shadow_price["datetime_ending_ept"].max(),
        burnham_shadow_price["datetime_ending_ept"].max()
    ),
    freq="h"
)

chicago_filled = (
    chicago_shadow_price
    .set_index("datetime_ending_ept")
    .reindex(full_range)
    .fillna(0)
    .rename_axis("datetime_ending_ept")
    .reset_index()
)

burnham_filled = (
    burnham_shadow_price
    .set_index("datetime_ending_ept")
    .reindex(full_range)
    .fillna(0)
    .rename_axis("datetime_ending_ept")
    .reset_index()
)


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=chicago_filled["datetime_ending_ept"],
    y=chicago_filled["shadow_price"],
    name="Chicago",
    mode="lines"
))

fig.add_trace(go.Scatter(
    x=burnham_filled["datetime_ending_ept"],
    y=burnham_filled["shadow_price"],
    name="Burnham",
    mode="lines",
    line=dict(dash="dash")
))

fig.update_layout(
    title="Chicago vs Burnham Shadow Price",
    hovermode="x unified"
)

fig.show()

## county

In [31]:
# Get all nodenames that belong to agg_nodename == "COMED" in hourly_load
ce_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "COMED", "nodename"]
    .dropna()
    .unique()
)

# Map those COMED nodenames to county_name using node_cty_map
ce_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(ce_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those COMED nodenames
ce_counties = ce_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_ce_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(ce_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_ce_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_ce_counties)
].copy()

In [32]:
hourly_load_county = hourly_load_ce_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

county_load = county_load[county_load["state_short_name"] == "IL"].copy()

In [33]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [34]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [35]:
county_load.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude
0,2026-04-01,IL_Boone,IL,110.37109,42.29,-88.89
1,2026-04-01,IL_Bureau,IL,7.32962,41.53,-89.71
2,2026-04-01,IL_Cook,IL,4873.16736,41.88,-87.80
3,2026-04-01,IL_DeKalb,IL,182.88795,41.89,-88.76
4,2026-04-01,IL_DuPage,IL,1087.61351,41.84,-88.09


In [36]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    burnham_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [37]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [38]:
df = county_load_shadow.copy()

# feature_cols = ["distributed_load_mw", "temperature_2m", "wind_speed_10m"] 
feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
7,IL_Kendall,0.260259
12,IL_McHenry,0.249055
5,IL_Kane,0.234988
16,IL_Will,0.233739
6,IL_Kankakee,0.227265
10,IL_Lee,0.220136
17,IL_Winnebago,0.217899
0,IL_Boone,0.210324
4,IL_Grundy,0.209618
2,IL_DeKalb,0.197742


In [39]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [40]:
df = county_load_shadow_price.copy()

node = "IL_Will"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# Combined look

In [ ]:
# rename for alignment
chicago = chicago_shadow_price.rename(columns={"shadow_price": "chicago_shadow_price"})
burnham = burnham_shadow_price.rename(columns={"shadow_price": "burnham_shadow_price"})

# merge
cook = pd.merge(
    chicago[["datetime_ending_ept", "chicago_shadow_price"]],
    burnham[["datetime_ending_ept", "burnham_shadow_price"]],
    on="datetime_ending_ept",
    how="outer"
)

# fill missing with 0
cook = cook.fillna(0)

# sum
cook["shadow_price"] = (
    cook["chicago_shadow_price"] + cook["burnham_shadow_price"]
)

# match original structure
cook_shadow_price = cook[["datetime_ending_ept", "shadow_price"]].copy()
cook_shadow_price["nodename"] = "Cook"

# reorder columns to match your format
cook_shadow_price = cook_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values("datetime_ending_ept").reset_index(drop=True)

cook_shadow_price.head()

In [ ]:
hourly_load_county = hourly_load_ce_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

county_load = county_load[county_load["state_short_name"] == "IL"].copy()

In [ ]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [ ]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [ ]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    cook_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [ ]:
county_load_shadow_price.head()

In [ ]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [ ]:
df = county_load_shadow.copy()

# feature_cols = ["distributed_load_mw", "temperature_2m", "wind_speed_10m"] 
feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:
df = county_load_shadow_price.copy()

node = "IL_Cook"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
sub

In [ ]:
fig = px.scatter(
    sub,
    x="distributed_load_mw",
    y="shadow_price",
    title="Distributed Load vs Shadow Price (Cook County)",
    labels={
        "distributed_load_mw": "Distributed Load (MW)",
        "shadow_price": "Shadow Price"
    }
)

fig.show()

In [ ]:
sub["load_spread"] = (
    sub["distributed_load_mw"].shift(-1) - sub["distributed_load_mw"]
)

fig = px.scatter(
    sub,
    x="load_spread",
    y="shadow_price",
    title="Load Spread vs Shadow Price (Cook County)",
    labels={
        "load_spread": "Load Spread (MW)",
        "shadow_price": "Shadow Price"
    }
)

fig.show()

In [ ]:
positive_sp = sub[sub["shadow_price"] > 0].copy()

positive_sp["hour"] = positive_sp["datetime_ending_ept"].dt.hour

positive_sp[["datetime_ending_ept", "hour", "shadow_price"]]

hour_counts = (
    positive_sp["datetime_ending_ept"]
    .dt.hour
    .value_counts()
    .sort_index()
)

# conditional probability of binding given load bucket
pd.crosstab(
    pd.cut(sub["distributed_load_mw"], bins=10),
    sub["datetime_ending_ept"].dt.hour,
    values=(sub["shadow_price"] > 0),
    aggfunc="mean"
)

In [ ]:
heatmap_df = pd.crosstab( 
    pd.cut(sub["distributed_load_mw"], bins=10), 
    sub["datetime_ending_ept"].dt.hour, 
    values=(sub["shadow_price"] > 0), 
    aggfunc="mean" 
    )

heatmap_df.index = heatmap_df.index.astype(str)

count_df = pd.crosstab(
    pd.cut(sub["distributed_load_mw"], bins=10),
    sub["datetime_ending_ept"].dt.hour
)

count_df.index = count_df.index.astype(str)
count_df = count_df.loc[heatmap_df.index, heatmap_df.columns]

import plotly.express as px

fig = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="RdYlBu_r",
    template="plotly_white",
    labels=dict(
        x="Hour Ending",
        y="Load Bucket",
        color="Binding Probability"
    ),
    title="Binding Probability Heatmap (Load vs Hour)"
)

# attach counts to hover
fig.update_traces(
    customdata=count_df.values,
    hovertemplate=
        "Hour: %{x}<br>" +
        "Load: %{y}<br>" +
        "Binding Prob: %{z:.2f}<br>" +
        "Count: %{customdata}<extra></extra>"
)

fig.show()


In [ ]:
heatmap_df = pd.crosstab(
    pd.cut(sub["load_spread"], bins=6),
    sub["datetime_ending_ept"].dt.hour,
    values=(sub["shadow_price"] > 0),
    aggfunc="mean"
)

import plotly.express as px

heatmap_df.index = heatmap_df.index.astype(str)

fig = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="RdYlBu_r",
    labels=dict(
        x="Hour Ending",
        y="Load Bucket",
        color="Binding Probability"
    ),
    title="Binding Probability Heatmap (Load vs Hour)"
)

fig.show()

## data-driven threshold

In [ ]:
county_load_shadow_price

In [ ]:
cook_load_shadow = county_load_shadow_price[county_load_shadow_price["county_name"] == "IL_Cook"].copy()
cook_load_shadow = cook_load_shadow[["datetime_ending_ept", "county_name", "distributed_load_mw", "shadow_price"]]
cook_load_shadow

In [ ]:
import numpy as np
import pandas as pd

def find_low_load_threshold(df, county="IL_Cook"):
    
    df_node = df[df["county_name"] == county].copy()
    df_node = df_node.sort_values("datetime_ending_ept")
    
    # Define target
    df_node["positive_sp"] = (df_node["shadow_price"] > 0).astype(int)
    
    # Candidate thresholds (quantile grid)
    thresholds = np.percentile(
        df_node["distributed_load_mw"],
        np.linspace(5, 95, 50)
    )
    
    results = []
    
    for T in thresholds:
        subset = df_node[df_node["distributed_load_mw"] < T]
        
        if len(subset) < 30:
            continue
        
        prob = subset["positive_sp"].mean()
        count = len(subset)
        
        results.append({
            "threshold": T,
            "prob_positive_sp": prob,
            "count": count
        })
    
    res_df = pd.DataFrame(results)
    
    # Rank by probability, but avoid tiny samples
    res_df = res_df.sort_values(
        ["prob_positive_sp", "count"],
        ascending=[False, False]
    )
    
    best = res_df.iloc[0]
    
    return best, res_df

In [ ]:
best, all_thresholds = find_low_load_threshold(county_load_shadow_price)

print(best)

# COLLEGEC

COLLEGEC-COLLINSV TIE B 138 KV

## shadow

In [41]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("COLLEGEC")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
collegec_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
collegec_shadow_price = collegec_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
collegec_shadow_price["hour_ending"] = collegec_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
collegec_shadow_price["datetime_ending_ept"] = (
    collegec_shadow_price["file_date"] + pd.to_timedelta(collegec_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
collegec_shadow_price = collegec_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
collegec_shadow_price = collegec_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

collegec_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-04 14:00:00,COLLEGEC-COLLINSV TIE B 138 KV,112.28
1,2026-04-04 15:00:00,COLLEGEC-COLLINSV TIE B 138 KV,1500.00
2,2026-04-04 16:00:00,COLLEGEC-COLLINSV TIE B 138 KV,754.81
3,2026-04-04 17:00:00,COLLEGEC-COLLINSV TIE B 138 KV,831.03
4,2026-04-14 13:00:00,COLLEGEC-COLLINSV TIE B 138 KV,200.15


In [42]:
collegec_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [43]:
# Get all nodenames that belong to agg_nodename == "DAY" in hourly_load
aep_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP", "DAY", "ATSI"]), "nodename"]
    .dropna()
    .unique()
)

# Map those AEP nodenames to county_name using node_cty_map
aep_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(aep_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those AEP nodenames
aep_counties = aep_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_aep_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(aep_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_aep_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_aep_counties)
].copy()

In [44]:
hourly_load_county = hourly_load_aep_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [45]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [46]:
county_load = county_load[county_load["state_short_name"] == "OH"].copy()

In [47]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-04")].copy()

In [48]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    collegec_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

county_load_shadow = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

In [49]:
df = county_load_shadow_price.copy()

feature_cols = ["distributed_load_mw"] 

# correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
8,OH_Carroll,0.277708
64,OH_Preble,0.260264
32,OH_Highland,0.257388
11,OH_Clinton,0.243815
65,OH_Putnam,0.235805
...,...,...
84,OH_Wyandot,0.009265
4,OH_Athens,-0.013153
82,OH_Williams,-0.014112
60,OH_Perry,-0.058386


In [50]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

### plot

In [51]:
df = county_load_shadow_price.copy()

node = "OH_Preble"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [52]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [53]:
# node1 = "OH_Meigs" 
# node2 = "OH_Monroe" 

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node

In [54]:
collegec_hourly_load = hourly_load[
    (hourly_load["agg_nodename"] == "DAY") &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 20
    )
].copy()

In [ ]:
collegec_hourly_load = collegec_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

collegec_hourly_load = collegec_hourly_load[collegec_hourly_load["state_short_name"] == "OH"].copy()

In [ ]:
collegec_hourly_load = collegec_hourly_load[(collegec_hourly_load["datetime_ending_ept"] >= "2026-04-04")].copy()

In [ ]:
collegec_merged = collegec_hourly_load.merge(
    collegec_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

collegec_merged["shadow_price"] = collegec_merged["shadow_price"].fillna(0)

# total unique timestamps
total_ts = collegec_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    collegec_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
collegec_merged = collegec_merged[collegec_merged['nodename'].isin(valid_nodes)]

collegec_shadow = collegec_merged[
    (collegec_merged["shadow_price"].notna()) &
    (collegec_merged["shadow_price"] != 0)
].copy()

In [ ]:
df = collegec_merged.copy()

feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False).head(20)

In [ ]:
df_plot = collegec_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "GARAGERD69 KV   BK-2"

df = collegec_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
df = collegec_merged.copy()

# Pivot: rows = timestamp, columns = nodename
pivot = df.pivot_table(
    index="datetime_ending_ept",
    columns="nodename",
    values="distributed_load_mw"
)

import itertools

diff_dict = {}

for n1, n2 in itertools.combinations(pivot.columns, 2):
    diff_name = f"{n1}__minus__{n2}"
    diff_dict[diff_name] = pivot[n1] - pivot[n2]

diff_df = pd.DataFrame(diff_dict)

shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

merged = diff_df.join(shadow, how="inner")

corr = merged.corr()["shadow_price"].drop("shadow_price")

corr = corr.sort_values(ascending=False)
corr


In [ ]:
node1 = "MINSTER 69 KV   LOAD" 
node2 = "TAIT    12.5 KV LOAD" 

sub = collegec_merged[
    collegec_merged["nodename"].isin([node1, node2])
].copy()

sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# pivot to compute difference
pivot = sub.pivot_table(
    index="datetime_ending_ept",
    columns="nodename",
    values="distributed_load_mw"
)

# compute spread
spread = pivot[node1] - pivot[node2]
spread.name = "spread_mw"

shadow = collegec_merged[
    collegec_merged["nodename"] == node1
][["datetime_ending_ept", "shadow_price"]].copy()

shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
df_plot = df_plot.sort_values("datetime_ending_ept")

import plotly.graph_objects as go

fig = go.Figure()

# Left axis: spread
fig.add_trace(
    go.Scatter(
        x=df_plot["datetime_ending_ept"],
        y=df_plot["spread_mw"],
        name=f"{node1} - {node2} (MW)",
        yaxis="y1"
    )
)

# Right axis: shadow price
fig.add_trace(
    go.Scatter(
        x=df_plot["datetime_ending_ept"],
        y=df_plot["shadow_price"],
        name="Shadow Price",
        yaxis="y2"
    )
)

fig.update_layout(
    title=f"Spread vs Shadow Price<br>{node1} - {node2}",
    xaxis=dict(title="Datetime"),

    yaxis=dict(
        title="Spread (MW)",
        side="left"
    ),

    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),

    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# DAI - FREMNT

## shadow 

In [45]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("FREMNT")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
fremnt_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
fremnt_shadow_price = fremnt_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
fremnt_shadow_price["hour_ending"] = fremnt_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
fremnt_shadow_price["datetime_ending_ept"] = (
    fremnt_shadow_price["file_date"] + pd.to_timedelta(fremnt_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
fremnt_shadow_price = fremnt_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
fremnt_shadow_price = fremnt_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

fremnt_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-02 19:00:00,FREMNT-FREMONTC A 138 KV,166.67
1,2026-04-02 20:00:00,FREMNT-FREMONTC A 138 KV,1210.65
2,2026-04-02 21:00:00,FREMNT-FREMONTC A 138 KV,1322.57
3,2026-04-02 22:00:00,FREMNT-FREMONTC A 138 KV,538.15
4,2026-04-02 23:00:00,FREMNT-FREMONTC A 138 KV,90.84


In [46]:
fremnt_shadow_price[fremnt_shadow_price["nodename"] == "FREMNT-FREMONTC"]

,datetime_ending_ept,nodename,shadow_price


In [47]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("FREMNT")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
fremnt_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
fremnt_shadow_price_DA = fremnt_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
fremnt_shadow_price_DA["hour_ending"] = fremnt_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
fremnt_shadow_price_DA["datetime_ending_ept"] = (
    fremnt_shadow_price_DA["file_date"] + 
    pd.to_timedelta(fremnt_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
fremnt_shadow_price_DA = fremnt_shadow_price_DA.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename", "shadow_price": "shadow_price_DA"}
)

# reorder columns
fremnt_shadow_price_DA = fremnt_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

fremnt_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-01 07:00:00,FREMNT_138KVFRE-FRE1_1_LN,54.03
1,2026-04-01 09:00:00,FREMNT_138KVFRE-FRE1_1_LN,12.51
2,2026-04-01 14:00:00,FREMNT_138KVFRE-FRE1_1_LN,48.43
3,2026-04-01 16:00:00,FREMNT_138KVFRE-FRE1_1_LN,123.44
4,2026-04-01 19:00:00,FREMNT_138KVFRE-FRE1_1_LN,54.87


In [48]:
fremnt_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [49]:
# Get all nodenames that belong to agg_nodename == "ATSI" in hourly_load
atsi_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP", "ATSI"]), "nodename"]
    .dropna()
    .unique()
)

# Map those ATSI nodenames to county_name using node_cty_map
atsi_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(atsi_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those ATSI nodenames
atsi_counties = atsi_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_atsi_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(atsi_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_atsi_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_atsi_counties)
].copy()

In [50]:
hourly_load_county = hourly_load_atsi_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [51]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [52]:
county_load.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude
0,2026-04-01,IN_Adams,IN,30.89586,40.83,-85.03
1,2026-04-01,IN_Allen,IN,516.85976,41.07,-85.14
2,2026-04-01,IN_Blackford,IN,35.32250,40.42,-85.33
3,2026-04-01,IN_DeKalb,IN,338.94799,41.35,-84.95
4,2026-04-01,IN_Delaware,IN,146.10947,40.24,-85.43


In [53]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load = county_load[county_load["state_short_name"] == "OH"].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

county_load_shadow_price = county_load.merge(
    fremnt_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [54]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [55]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
24,OH_Henry,0.335009
4,OH_Auglaize,0.275313
54,OH_Ross,0.226603
56,OH_Scioto,0.221443
42,OH_Monroe,0.189438
...,...,...
50,OH_Pike,-0.030042
0,OH_Adams,-0.079703
19,OH_Gallia,-0.088915
52,OH_Putnam,-0.122919


In [56]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

### plot

In [57]:
county_load_shadow_price = county_load_shadow_price.merge(
    fremnt_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [58]:

df = county_load_shadow_price.copy()

node = "OH_Henry"

sub = df[df["county_name"] == node].copy()

sub = sub[sub["datetime_ending_ept"] >= "2026-04-12"].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.90, y=0.99),
    height=500,
    width=2000
)

fig.show()

### DA

In [59]:
county_load_shadow_price.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude,shadow_price,shadow_price_DA
0,2026-04-01,OH_Adams,OH,33.01097,38.84,-83.54,0.0,0.0
1,2026-04-01,OH_Allen,OH,285.97935,40.75,-84.14,0.0,0.0
2,2026-04-01,OH_Ashland,OH,37.16403,40.90,-82.34,0.0,0.0
3,2026-04-01,OH_Ashtabula,OH,196.28593,41.84,-80.76,0.0,0.0
4,2026-04-01,OH_Athens,OH,14.86627,39.36,-82.08,0.0,0.0


In [60]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price_DA"].notna()) &
    (county_load_shadow_price["shadow_price_DA"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

county_load_shadow['distributed_load_mw'] = county_load_shadow['distributed_load_mw'] - 45

df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price_DA"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False).head(20)

,county_name,distributed_load_mw
16,OH_Fairfield,0.501450
14,OH_Delaware,0.474843
56,OH_Scioto,0.467006
42,OH_Monroe,0.409768
17,OH_Franklin,0.408220
31,OH_Knox,0.392239
7,OH_Carroll,0.384346
45,OH_Noble,0.381178
34,OH_Licking,0.376274
24,OH_Henry,0.370417


In [61]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [62]:
# node1 = "OH_Belmont"
# node2 = "OH_Henry"

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node

In [ ]:
hourly_load_county[hourly_load_county["county_name"] == "OH_Henry"]['nodename'].unique()

In [ ]:
fremnt_hourly_load = hourly_load[
    (hourly_load["agg_nodename"] == "ATSI") &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 20
    )
].copy()

In [ ]:
fremnt_hourly_load = fremnt_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

In [ ]:
fremnt_hourly_load = fremnt_hourly_load[(fremnt_hourly_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [ ]:
fremnt_merged = fremnt_hourly_load.merge(
    fremnt_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

fremnt_merged["shadow_price"] = fremnt_merged["shadow_price"].fillna(0)

# total unique timestamps
total_ts = fremnt_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    fremnt_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
fremnt_merged = fremnt_merged[fremnt_merged['nodename'].isin(valid_nodes)]

# row-level filter
fremnt_shadow = fremnt_merged[
    (fremnt_merged["shadow_price"].notna()) &
    (fremnt_merged["shadow_price"] != 0)
].copy()

In [ ]:
df = fremnt_shadow.copy()

feature_cols = ["distributed_load_mw"]

# correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:
df_plot = fremnt_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "CMPBL_SP69 KV   TR2"

df = fremnt_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
# df = fremnt_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr


In [ ]:

# node1 = "CLARKAVE69 KV   ECMLOAD" 
# node2 = "GRFBD_TP69 KV   GRIEFBRD"

# sub = fremnt_merged[
#     fremnt_merged["nodename"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = fremnt_merged[
#     fremnt_merged["nodename"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

# DAI - HYATTCS

## shadow

In [9]:
RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("HYATTCS-MARYSVI2 MAR-HYA2   B  345 KV")].head()

,file_date,MARKET_TYPE_CODE,MONITORED_FACILITY_DESC,CONTINGENCY_FACILITY_DESC,SHADOW_PRICE,INTF_Name,IsNew,1,2,3,...,16,17,18,19,20,21,22,23,24,VALUE_DATETIME
145,2026-04-12,RT,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,LINE 345 KV MARYSVI2-TANGY TIE;MARYSVI2 345 KV...,-2000.000000,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV L/O LINE...,New,NaN,NaN,NaN,...,NaN,NaN,579.32,133.35,NaN,4.99,2.79,NaN,NaN,2026-04-12 17:35:00
177,2026-04-13,RT,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,LINE 345 KV MARYSVI2-TANGY TIE;MARYSVI2 345 KV...,-2000.000000,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV L/O LINE...,Old,NaN,NaN,NaN,...,169.97,440.85,309.26,213.57,582.23,52.37,NaN,NaN,NaN,2026-04-13 19:20:00
201,2026-04-14,RT,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,LINE 345 KV MARYSVI2-TANGY TIE;MARYSVI2 345 KV...,-2000.000000,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV L/O LINE...,Old,NaN,NaN,NaN,...,356.25,805.78,1176.24,1608.25,2000.00,2000.00,321.69,NaN,NaN,2026-04-14 19:00:00
235,2026-04-15,RT,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,LINE 345 KV MARYSVI2-TANGY TIE;MARYSVI2 345 KV...,-2000.000000,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV L/O LINE...,Old,NaN,NaN,NaN,...,1783.06,1982.88,1988.10,2000.00,2000.00,1942.24,666.67,NaN,NaN,2026-04-15 19:10:00
253,2026-04-16,RT,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,LINE 345 KV MARYSVI2-TANGY TIE;MARYSVI2 345 KV...,-1080.530029,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV L/O LINE...,Old,NaN,NaN,NaN,...,324.61,96.94,116.25,104.52,NaN,NaN,NaN,NaN,NaN,2026-04-16 14:50:00


In [10]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "HYATTCS-MARYSVI2 MAR-HYA2   B  345 KV"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
hyattcs_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
hyattcs_shadow_price = hyattcs_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
hyattcs_shadow_price["hour_ending"] = hyattcs_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
hyattcs_shadow_price["datetime_ending_ept"] = (
    hyattcs_shadow_price["file_date"] + pd.to_timedelta(hyattcs_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
hyattcs_shadow_price = hyattcs_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
hyattcs_shadow_price = hyattcs_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

hyattcs_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-12 18:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,579.32
1,2026-04-12 19:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,133.35
2,2026-04-12 21:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,4.99
3,2026-04-12 22:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,2.79
4,2026-04-13 14:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,194.60


In [11]:
hyattcs_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [12]:
# Get all nodenames that belong to agg_nodename == "AEP" in hourly_load
aep_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP", "DAY", "ATSI"]), "nodename"]
    .dropna()
    .unique()
)

# Map those AEP nodenames to county_name using node_cty_map
aep_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(aep_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those AEP nodenames
aep_counties = aep_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_aep_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(aep_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_aep_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_aep_counties)
].copy()

In [13]:
hourly_load_county = hourly_load_aep_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [14]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [15]:
county_load = county_load[county_load["state_short_name"] == "OH"].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-12")].copy()

In [16]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    hyattcs_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [17]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [18]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
51,OH_Monroe,0.656415
18,OH_Delaware,0.634844
11,OH_Clinton,0.628943
57,OH_Ottawa,0.596003
65,OH_Richland,0.583165
...,...,...
33,OH_Holmes,0.060271
0,OH_Adams,0.038386
61,OH_Pike,0.023333
66,OH_Ross,0.022324


In [19]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [20]:
df = county_load_shadow_price.copy()

node = "OH_Delaware"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [21]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [22]:
# node1 = "OH_Brown"
# node2 = "OH_Wood"

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node

In [23]:
hyattcs_hourly_load = hourly_load[
    (hourly_load["agg_nodename"].isin(["AEP", "DAY", "ATSI"])) &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 20
    )
].copy()

In [24]:
hyattcs_hourly_load = hyattcs_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")


hyattcs_hourly_load = hyattcs_hourly_load[hyattcs_hourly_load["state_short_name"] == "OH"].copy()

In [25]:
hyattcs_hourly_load = hyattcs_hourly_load[(hyattcs_hourly_load["datetime_ending_ept"] >= "2026-04-13")].copy()

In [26]:
hyattcs_merged = hyattcs_hourly_load.merge(
    hyattcs_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

hyattcs_merged["shadow_price"] = hyattcs_merged["shadow_price"].fillna(0)


In [27]:
# total unique timestamps
total_ts = hyattcs_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    hyattcs_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
hyattcs_merged = hyattcs_merged[hyattcs_merged['nodename'].isin(valid_nodes)]

In [28]:
hyattcs_shadow = hyattcs_merged[
    (hyattcs_merged["shadow_price"].notna()) &
    (hyattcs_merged["shadow_price"] != 0)
].copy()

In [29]:
df = hyattcs_shadow.copy()

feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False).head(20)

,nodename,distributed_load_mw
308,NWILLARD69 KV LOAD,0.715540
401,WILDCREE138 KV T1,0.695022
282,NATRIUM2138 KV DOMENRGY,0.662098
249,LAKVEW 34 KV LAKVW LD,0.642414
56,AIR_PROD138 KV TR2,0.620501
402,WILMING269 KV BK-1,0.608407
269,MAYFLD 34 KV E,0.598181
403,WILMING269 KV COLSTBK1,0.571382
86,BLACKRIV69 KV WILLOW,0.569737
55,AIR_PROD138 KV TR1,0.568012


In [30]:
df_plot = hyattcs_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [31]:
node = "WILDCREE138 KV  T1"

df = hyattcs_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [32]:
# df = hyattcs_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr


In [33]:
# node1 = "COSGRAY 345 KV  CMH0914" 
# node2 = "JUGSTRET138 KV  T9"

# sub = hyattcs_merged[
#     hyattcs_merged["nodename"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="nodename",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = hyattcs_merged[
#     hyattcs_merged["nodename"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

# DAI - PIERCE

## shadow

In [ ]:
RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "PIERCE-DEARBORN             B  345 KV"].head()

In [ ]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "PIERCE-DEARBORN             B  345 KV"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
pierce_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
pierce_shadow_price = pierce_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
pierce_shadow_price["hour_ending"] = pierce_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
pierce_shadow_price["datetime_ending_ept"] = (
    pierce_shadow_price["file_date"] + pd.to_timedelta(pierce_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
pierce_shadow_price = pierce_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
pierce_shadow_price = pierce_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

pierce_shadow_price.head()

In [ ]:
pierce_shadow_price["shadow_price"].max()

## county

In [ ]:
# Get all nodenames that belong to agg_nodename == "DEOK" in hourly_load
deok_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["DEOK", "EKPC"]), "nodename"]
    .dropna()
    .unique()
)

# Map those DEOK nodenames to county_name using node_cty_map
deok_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(deok_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DEOK nodenames
deok_counties = deok_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_deok_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(deok_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_deok_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_deok_counties)
].copy()

In [ ]:
hourly_load_county = hourly_load_deok_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [ ]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [ ]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-05")].copy()

In [ ]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    pierce_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [ ]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [ ]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:

df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [ ]:
df = county_load_shadow_price.copy()

node = "KY_Boone"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [ ]:
# node1 = "KY_Campbell"
# node2 = "OH_Brown"

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node

In [ ]:
pierce_hourly_load = hourly_load[
    (hourly_load["agg_nodename"].isin(["DEOK", "EKPC"])) &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 10
    )
].copy()

In [ ]:
pierce_hourly_load = pierce_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

In [ ]:
pierce_hourly_load = pierce_hourly_load[pierce_hourly_load["state_short_name"] == "KY"].copy()

pierce_hourly_load = pierce_hourly_load[(pierce_hourly_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [ ]:
pierce_merged = pierce_hourly_load.merge(
    pierce_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

pierce_merged["shadow_price"] = pierce_merged["shadow_price"].fillna(0)

# total unique timestamps
total_ts = pierce_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    pierce_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
pierce_merged = pierce_merged[pierce_merged['nodename'].isin(valid_nodes)]

# row-level filter
pierce_shadow = pierce_merged[
    (pierce_merged["shadow_price"].notna()) &
    (pierce_merged["shadow_price"] != 0)
].copy()

In [ ]:
df = pierce_shadow.copy()

feature_cols = ["distributed_load_mw"]

# correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False).head(20)

In [ ]:
df_plot = pierce_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "08VILLA169 KV   BK1"

df = pierce_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
df = pierce_shadow.copy()

# Pivot: rows = timestamp, columns = nodename
pivot = df.pivot_table(
    index="datetime_ending_ept",
    columns="nodename",
    values="distributed_load_mw"
)

import itertools

diff_dict = {}

for n1, n2 in itertools.combinations(pivot.columns, 2):
    diff_name = f"{n1}__minus__{n2}"
    diff_dict[diff_name] = pivot[n1] - pivot[n2]

diff_df = pd.DataFrame(diff_dict)

shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

merged = diff_df.join(shadow, how="inner")

corr = merged.corr()["shadow_price"].drop("shadow_price")

corr = corr.sort_values(ascending=False)
corr


In [ ]:

node1 = "CEDARVIL138 KV  BK1" 
node2 = "CLINTONC138 KV  BK1"

sub = pierce_merged[
    pierce_merged["nodename"].isin([node1, node2])
].copy()

sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# pivot to compute difference
pivot = sub.pivot_table(
    index="datetime_ending_ept",
    columns="nodename",
    values="distributed_load_mw"
)

# compute spread
spread = pivot[node1] - pivot[node2]
spread.name = "spread_mw"

shadow = pierce_merged[
    pierce_merged["nodename"] == node1
][["datetime_ending_ept", "shadow_price"]].copy()

shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
df_plot = df_plot.sort_values("datetime_ending_ept")

import plotly.graph_objects as go

fig = go.Figure()

# Left axis: spread
fig.add_trace(
    go.Scatter(
        x=df_plot["datetime_ending_ept"],
        y=df_plot["spread_mw"],
        name=f"{node1} - {node2} (MW)",
        yaxis="y1"
    )
)

# Right axis: shadow price
fig.add_trace(
    go.Scatter(
        x=df_plot["datetime_ending_ept"],
        y=df_plot["shadow_price"],
        name="Shadow Price",
        yaxis="y2"
    )
)

fig.update_layout(
    title=f"Spread vs Shadow Price<br>{node1} - {node2}",
    xaxis=dict(title="Datetime"),

    yaxis=dict(
        title="Spread (MW)",
        side="left"
    ),

    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),

    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# DAI - Batesville

### shadow

In [ ]:
RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "Batesville-Hubble 138 l/o Tanners Crk-Miami Fort 345"].head()

In [ ]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "Batesville-Hubble 138 l/o Tanners Crk-Miami Fort 345"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
batesville_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
batesville_shadow_price = batesville_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
batesville_shadow_price["hour_ending"] = batesville_shadow_price["hour_ending"].astype(int)

# set operating date
# replace with the actual DA operating date if needed
operating_date = pd.Timestamp("2026-04-21")

# build datetime_ending_ept
batesville_shadow_price["datetime_ending_ept"] = (
    batesville_shadow_price["file_date"] + pd.to_timedelta(batesville_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
batesville_shadow_price = batesville_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
batesville_shadow_price = batesville_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

batesville_shadow_price.head()

In [ ]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"] == "Batesville-Hubble 138 l/o Tanners Crk-Miami Fort 345"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
batesville_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
batesville_shadow_price_DA = batesville_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
batesville_shadow_price_DA["hour_ending"] = batesville_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
batesville_shadow_price_DA["datetime_ending_ept"] = (
    batesville_shadow_price_DA["file_date"] + 
    pd.to_timedelta(batesville_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
batesville_shadow_price_DA = batesville_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
batesville_shadow_price_DA = batesville_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

batesville_shadow_price_DA.head()

In [ ]:
batesville_shadow_price["shadow_price"].max()

## county

In [ ]:
# Get all nodenames that belong to agg_nodename == "DEOK" in hourly_load
deok_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DEOK", "nodename"]
    .dropna()
    .unique()
)

# Map those DEOK nodenames to county_name using node_cty_map
deok_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(deok_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DEOK nodenames
deok_counties = deok_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_deok_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(deok_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_deok_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_deok_counties)
].copy()

In [ ]:
hourly_load_county = hourly_load_deok_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [ ]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [ ]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-05")].copy()

In [ ]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    batesville_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

county_load_shadow = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

In [ ]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:

df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [ ]:
county_load_shadow_price = county_load_shadow_price.merge(
    batesville_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)


df = county_load_shadow_price.copy()

node = "OH_Butler"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
# df = county_load_shadow.copy()

# # Pivot: rows = timestamp, columns = nodename
# pivot = df.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# import itertools

# diff_dict = {}

# for n1, n2 in itertools.combinations(pivot.columns, 2):
#     diff_name = f"{n1}__minus__{n2}"
#     diff_dict[diff_name] = pivot[n1] - pivot[n2]

# diff_df = pd.DataFrame(diff_dict)

# shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

# merged = diff_df.join(shadow, how="inner")

# corr = merged.corr()["shadow_price"].drop("shadow_price")

# corr = corr.sort_values(ascending=False)
# corr

In [ ]:
# node1 = "KY_Kenton" 
# node2 = "OH_Butler" 

# sub = county_load_shadow_price[
#     county_load_shadow_price["county_name"].isin([node1, node2])
# ].copy()

# sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# # pivot to compute difference
# pivot = sub.pivot_table(
#     index="datetime_ending_ept",
#     columns="county_name",
#     values="distributed_load_mw"
# )

# # compute spread
# spread = pivot[node1] - pivot[node2]
# spread.name = "spread_mw"

# shadow = county_load_shadow_price[
#     county_load_shadow_price["county_name"] == node1
# ][["datetime_ending_ept", "shadow_price"]].copy()

# shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
# shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

# df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
# df_plot = df_plot.sort_values("datetime_ending_ept")

# import plotly.graph_objects as go

# fig = go.Figure()

# # Left axis: spread
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["spread_mw"],
#         name=f"{node1} - {node2} (MW)",
#         yaxis="y1"
#     )
# )

# # Right axis: shadow price
# fig.add_trace(
#     go.Scatter(
#         x=df_plot["datetime_ending_ept"],
#         y=df_plot["shadow_price"],
#         name="Shadow Price",
#         yaxis="y2"
#     )
# )

# fig.update_layout(
#     title=f"Spread vs Shadow Price<br>{node1} - {node2}",
#     xaxis=dict(title="Datetime"),

#     yaxis=dict(
#         title="Spread (MW)",
#         side="left"
#     ),

#     yaxis2=dict(
#         title="Shadow Price",
#         overlaying="y",
#         side="right"
#     ),

#     legend=dict(x=0.01, y=0.99),
#     height=500,
#     width=2000
# )

# fig.show()

## node

In [ ]:
batesville_hourly_load = hourly_load[
    (hourly_load["agg_nodename"] == "DEOK") &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 20
    )
].copy()

In [ ]:
batesville_hourly_load = batesville_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")


In [ ]:
batesville_hourly_load = batesville_hourly_load[(batesville_hourly_load["datetime_ending_ept"] >= "2026-04-06")].copy()

In [ ]:
batesville_merged = batesville_hourly_load.merge(
    batesville_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

batesville_merged["shadow_price"] = batesville_merged["shadow_price"].fillna(0)

# total unique timestamps
total_ts = batesville_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    batesville_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
batesville_merged = batesville_merged[batesville_merged['nodename'].isin(valid_nodes)]

batesville_shadow = batesville_merged[
    (batesville_merged["shadow_price"].notna()) &
    (batesville_merged["shadow_price"] != 0)
].copy()

In [ ]:
df = batesville_shadow.copy()

feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:
df_plot = batesville_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "WILLEY  138 KV  BK3"

df = batesville_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

In [ ]:
df = batesville_shadow.copy()

# Pivot: rows = timestamp, columns = nodename
pivot = df.pivot_table(
    index="datetime_ending_ept",
    columns="nodename",
    values="distributed_load_mw"
)

import itertools

diff_dict = {}

for n1, n2 in itertools.combinations(pivot.columns, 2):
    diff_name = f"{n1}__minus__{n2}"
    diff_dict[diff_name] = pivot[n1] - pivot[n2]

diff_df = pd.DataFrame(diff_dict)

shadow = df.groupby("datetime_ending_ept")["shadow_price"].mean()

merged = diff_df.join(shadow, how="inner")

corr = merged.corr()["shadow_price"].drop("shadow_price")

corr = corr.sort_values(ascending=False)
corr


In [ ]:

node1 = "CITYHAML69 KV   69LD1" 
node2 = "CLINTONC138 KV  BK1"

sub = batesville_merged[
    batesville_merged["nodename"].isin([node1, node2])
].copy()

sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])

# pivot to compute difference
pivot = sub.pivot_table(
    index="datetime_ending_ept",
    columns="nodename",
    values="distributed_load_mw"
)

# compute spread
spread = pivot[node1] - pivot[node2]
spread.name = "spread_mw"

shadow = batesville_merged[
    batesville_merged["nodename"] == node1
][["datetime_ending_ept", "shadow_price"]].copy()

shadow["datetime_ending_ept"] = pd.to_datetime(shadow["datetime_ending_ept"])
shadow = shadow.set_index("datetime_ending_ept")["shadow_price"]

df_plot = pd.concat([spread, shadow], axis=1).dropna().reset_index()
df_plot = df_plot.sort_values("datetime_ending_ept")

import plotly.graph_objects as go

fig = go.Figure()

# Left axis: spread
fig.add_trace(
    go.Scatter(
        x=df_plot["datetime_ending_ept"],
        y=df_plot["spread_mw"],
        name=f"{node1} - {node2} (MW)",
        yaxis="y1"
    )
)

# Right axis: shadow price
fig.add_trace(
    go.Scatter(
        x=df_plot["datetime_ending_ept"],
        y=df_plot["shadow_price"],
        name="Shadow Price",
        yaxis="y2"
    )
)

fig.update_layout(
    title=f"Spread vs Shadow Price<br>{node1} - {node2}",
    xaxis=dict(title="Datetime"),

    yaxis=dict(
        title="Spread (MW)",
        side="left"
    ),

    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),

    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# BECO

## shadow 

In [ ]:
RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("BECO-PARAGNPK")].head()

In [ ]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "BECO-PARAGNPK 2207A         B  230 KV"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
beco_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
beco_shadow_price = beco_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
beco_shadow_price["hour_ending"] = beco_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
beco_shadow_price["datetime_ending_ept"] = (
    beco_shadow_price["file_date"] + pd.to_timedelta(beco_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
beco_shadow_price = beco_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
beco_shadow_price = beco_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

beco_shadow_price.head()

In [ ]:
beco_shadow_price["shadow_price"].max()

## county

In [ ]:
# Get all nodenames that belong to agg_nodename == "BGE" in hourly_load
bge_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["BGE", "PEPCO", "APS", "DOM"]), "nodename"]
    .dropna()
    .unique()
)

# Map those DEOK nodenames to county_name using node_cty_map
bge_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(bge_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those BGE nodenames
bge_counties = bge_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_bge_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(bge_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_bge_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_bge_counties)
].copy()

In [ ]:
hourly_load_county = hourly_load_bge_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [ ]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [ ]:
county_load = county_load[county_load["state_short_name"].isin(["MD", "VA"])].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-20")].copy()

In [ ]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    beco_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [ ]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [ ]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:

df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

### plot

In [ ]:
df = county_load_shadow_price.copy()

node = "VA_Fairfax"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

## node

In [ ]:
beco_hourly_load = hourly_load[
    (hourly_load["agg_nodename"].isin(["BGE", "PEPCO", "APS", "DOM"])) &
    (
        hourly_load.groupby("nodename")["distributed_load_mw"]
        .transform("max") > 30
    )
].copy()

In [ ]:
beco_hourly_load = beco_hourly_load[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

beco_hourly_load = beco_hourly_load[beco_hourly_load["state_short_name"].isin(["MD", "VA"])].copy()

In [ ]:
beco_hourly_load = beco_hourly_load[(beco_hourly_load["datetime_ending_ept"] >= "2026-04-20")].copy()

In [ ]:
beco_merged = beco_hourly_load.merge(
    beco_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

beco_merged["shadow_price"] = beco_merged["shadow_price"].fillna(0)


In [ ]:
# total unique timestamps
total_ts = beco_merged['datetime_ending_ept'].nunique()

# count unique timestamps per nodename
node_ts_counts = (
    beco_merged
    .groupby('nodename')['datetime_ending_ept']
    .nunique()
)

# filter nodenames meeting the 80% threshold
valid_nodes = node_ts_counts[node_ts_counts / total_ts > 0.8].index

# keep only those rows
beco_merged = beco_merged[beco_merged['nodename'].isin(valid_nodes)]

In [ ]:
beco_shadow = beco_merged[
    (beco_merged["shadow_price"].notna()) &
    (beco_merged["shadow_price"] != 0)
].copy()

In [ ]:
df = beco_shadow.copy()

feature_cols = ["distributed_load_mw"]

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("nodename")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False).head(20)

In [ ]:
df_plot = beco_merged.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="nodename",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="nodename",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="Node Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
node = "BATERYHT230 KV  TX1"

df = beco_merged.copy()

sub = df[df["nodename"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# NILES

## shadow

In [ ]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("NILES")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
niles_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
niles_shadow_price = niles_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
niles_shadow_price["hour_ending"] = niles_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
niles_shadow_price["datetime_ending_ept"] = (
    niles_shadow_price["file_date"] + pd.to_timedelta(niles_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
niles_shadow_price = niles_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
niles_shadow_price = niles_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

niles_shadow_price.head()

In [ ]:
niles_shadow_price["shadow_price"].max()

## county

In [ ]:
# Get all nodenames that belong to agg_nodename == "ATSI" in hourly_load
atsi_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["ATSI", "PENELEC"]), "nodename"]
    .dropna()
    .unique()
)

# Map those ATSI nodenames to county_name using node_cty_map
atsi_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(atsi_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those ATSI nodenames
atsi_counties = atsi_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_atsi_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(atsi_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_atsi_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_atsi_counties)
].copy()

In [ ]:
hourly_load_county = hourly_load_atsi_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [ ]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [ ]:
county_load = county_load[county_load["state_short_name"].isin(["OH", "PA"])].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-20")].copy()

In [ ]:
county_load.head()

In [ ]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    niles_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [ ]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [ ]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

In [ ]:
df_plot = county_load_shadow_price.merge(
    corr_by_node.rename(columns={"distributed_load_mw": "corr"}),
    on="county_name",
    how="left"
).dropna(subset=["latitude", "longitude", "corr"])

fig = px.scatter_map(
    df_plot,
    lat="latitude",
    lon="longitude",
    color="corr",
    hover_name="county_name",
    hover_data={
        "state_short_name": True,
        "latitude": False,
        "longitude": False,
        "corr": ':.3f'
    },
    zoom=5,
    height=700,
    title="County Correlation Map"
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

### plot

In [ ]:
df = county_load_shadow_price.copy()

node = "OH_Trumbull"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()